In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# Utility functions
# -----------------------------
def is_terminal(state):
    return all(h == 0 for h in state)

def get_moves(state):
    """Generate all possible successor states from current state."""
    moves = []
    for i, heap in enumerate(state):
        if heap > 0:
            for remove in range(1, heap + 1):
                new_state = list(state)
                new_state[i] -= remove
                moves.append((state, tuple(new_state)))
    return moves

# -----------------------------
# Alpha–Beta Pruning
# -----------------------------
explored_nodes = 0
pruned_nodes = 0

def alphabeta(state, alpha, beta, maximizing_player=True, graph=None, parent=None, best_path=None):
    global explored_nodes, pruned_nodes
    if graph is None:
        graph = nx.DiGraph()
    if best_path is None:
        best_path = {}

    explored_nodes += 1
    node_label = str(state)
    if parent is not None:
        graph.add_edge(str(parent), node_label)

    # Terminal state check
    if is_terminal(state):
        value = -1 if maximizing_player else 1
        print(f"Terminal state {state} → {value}")
        graph.nodes[node_label]["label"] = f"{state}\nVal={value}"
        return value, graph, best_path

    if maximizing_player:
        print(f"MAX turn at {state} | α={alpha}, β={beta}")
        value = float('-inf')
        best_child = None
        for move in get_moves(state):
            print(f"  Considering: {move[0]} → {move[1]}")
            child_val, graph, best_path = alphabeta(move[1], alpha, beta, False, graph, state, best_path)
            if child_val > value:
                value = child_val
                best_child = move[1]
            alpha = max(alpha, value)
            if alpha >= beta:  # prune
                pruned_nodes += 1
                print(f"  [Pruned branch at {move[1]} because α={alpha} ≥ β={beta}]")
                break
        graph.nodes[node_label]["label"] = f"{state}\nMAX={value}"
        if best_child:
            best_path[state] = best_child
        return value, graph, best_path

    else:
        print(f"MIN turn at {state} | α={alpha}, β={beta}")
        value = float('inf')
        best_child = None
        for move in get_moves(state):
            print(f"  Considering: {move[0]} → {move[1]}")
            child_val, graph, best_path = alphabeta(move[1], alpha, beta, True, graph, state, best_path)
            if child_val < value:
                value = child_val
                best_child = move[1]
            beta = min(beta, value)
            if alpha >= beta:  # prune
                pruned_nodes += 1
                print(f"  [Pruned branch at {move[1]} because α={alpha} ≥ β={beta}]")
                break
        graph.nodes[node_label]["label"] = f"{state}\nMIN={value}"
        if best_child:
            best_path[state] = best_child
        return value, graph, best_path

# -----------------------------
# Draw Game Tree (optional)
# -----------------------------
def draw_tree(graph, best_path, root):
    pos = nx.spring_layout(graph, seed=42)
    labels = nx.get_node_attributes(graph, "label")

    # Highlight best path
    path_edges = []
    current = root
    while current in best_path:
        nxt = best_path[current]
        path_edges.append((str(current), str(nxt)))
        current = nxt

    plt.figure(figsize=(12, 8))
    nx.draw(graph, pos, with_labels=False, node_size=2000, node_color="lightblue", edge_color="gray")
    nx.draw_networkx_labels(graph, pos, labels, font_size=8)
    nx.draw_networkx_edges(graph, pos, edgelist=path_edges, edge_color="red", width=2)
    plt.title("Nim Alpha–Beta Pruned Tree with Best Path Highlighted")
    plt.show()

# -----------------------------
# Example Run
# -----------------------------
if __name__ == "__main__":
    initial_state = (3, 4, 5)  # Example
    explored_nodes, pruned_nodes = 0, 0

    val, graph, best_path = alphabeta(initial_state, float('-inf'), float('inf'), True)

    print("\n=============================")
    print(f"Final Alpha–Beta Value from {initial_state} = {val}")
    if initial_state in best_path:
        print(f"Best Move for MAX: {initial_state} → {best_path[initial_state]}")
    else:
        print("No best move found.")
    print(f"Nodes Explored: {explored_nodes}")
    print(f"Nodes Pruned: {pruned_nodes}")

    # Optional visualization
    draw_tree(graph, best_path, initial_state)
